In [1]:
import matplotlib.pyplot as plt
import torch
import os
from pathlib import Path
os.chdir("..")  # Change root dir to project folder
os.getcwd()

'/hs/fs08/data/group-brueggen/tmartinez/diffusion/src'

# Animated Sampling Process

## Sampling

Set paths & load model:

In [ ]:
from datetime import datetime
from pathlib import Path

from models.sample import sample_batch
from utils.init_utils import (
    load_diffusion_from_folder, load_model_from_folder, load_snapshot
)
from utils.device_utils import distribute_model

# Directories
out_folder = Path(f"/home/bbd0953/diffusion/analysis_results/EDM_valFix/samples_EDM_valFix")
model_parent = Path("/home/bbd0953/diffusion/results/")

model_name = "EDM_valFix"
model_dir = model_parent / model_name
snapshot_iter = 0 # 20_000  # Set 0 for default

# Load model and diffusion
model = load_model_from_folder(model_dir)
if snapshot_iter:
    model = load_snapshot(model_dir, snapshot_iter, model=model)

diffusion = load_diffusion_from_folder(model_dir)

Prepare Sampling/Loading:

In [ ]:
# File name number formatting
def fmt(v):
    match v:
        case int() if not v % 1000:
            return f'{v / 1000:.0f}k'
        case int():
            return str(v)
        case float() if v >= 1:
            return f'{v:.2f}'
        case float():
            return f'{v:.2e}'

# Sampling setup
n_devices = 1
diffusion.timesteps = 25
batch_size = 1000 * n_devices # Batch size
n_batches = 1

sample_kwargs = {
    'sigma_min': 2e-3,
}

outfile_specifier = (
    f"{batch_size * n_batches}Imgs_T={diffusion.timesteps}"\
    f"{'_' * bool(len(sample_kwargs)) + '_'.join(f'{k}={fmt(v)}' for k, v in sample_kwargs.items())}"
)
if snapshot_iter:
    outfile_specifier = f"itr={fmt(snapshot_iter)}_" + outfile_specifier

# Output file for samples
out_file = (
    out_folder /
    f"{model_dir.name}_samples_{outfile_specifier}.pt"
)

Sample:

In [ ]:
# Prepare time measurement
dt = lambda t0: datetime.now() - t0

# Distribute model
model, device_ids = distribute_model(model, n_devices)

# Sample from model
batch_list = []
t0 = datetime.now()
for _ in range(n_batches):
    batch = sample_batch(model, diffusion, 
                         bsize=batch_size,
                         return_steps=True, **sample_kwargs)
    batch_list.append(batch)

runtime = dt(t0)
print(f"Runtime {runtime}")

model = (
    model.module.to('cpu') if isinstance(model, torch.nn.DataParallel)
    else model.to('cpu')
)


# Output of sample_batch is list with T+1 entries of shape
# (bsize, 1, 80, 80).
# Batch_list is a list of such lists with n_batches entries,
# i.e. n_batches x (T+1) x (bsize, 1, 80, 80).
# We want it as a single tensor of shape
# (n_batches * bsize, T+1, 1, 80, 80).
batch_st = torch.concat([
    torch.stack(b, dim=1) for b in batch_list
]).cpu()

# Scale images from [-1, 1] to [0, 1]
batch_st = (batch_st + 1) / 2
torch.save(batch_st, out_file)

Load Samples:

In [ ]:
batch_st = torch.load(out_file, map_location='cpu')

## Distributions

Functions for getting statistics & running animation:

In [ ]:
import pickle
from collections import OrderedDict
from importlib import reload

import numpy as np
from tqdm import tqdm
from utils.stats_utils import norm, norm_err, centers
from analysis.image_metrics import image_mean, active_pixels, batch_metric
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from utils.plot_utils import add_distribution_plot
import analysis.model_evaluations as meval
reload(meval)
from analysis.model_evaluations import get_distributions_lofar

def get_metrics_and_distributions(batch_st, pickle_file=None):

    if pickle_file is not None and pickle_file.exists():
        print(f"Loading metrics and distributions from {pickle_file}.")
        with open(pickle_file, 'rb') as f:
            return pickle.load(f)

    # Those will be called in the loop:
    funcs_and_bins = {
        'Image_Mean': (image_mean, np.linspace(0, 1, 257)),
        'Active_Pixels': (active_pixels, np.linspace(0, 6400, 129)),
    }
    # Split into two dictionaries
    metric_funcs, bins_dict = [
        {key: val[i] for key, val in funcs_and_bins.items()}
        for i in range(len(next(iter(funcs_and_bins.values()))))
    ]
    bins_dict['Pixel_Intensity'] = np.linspace(0, 1, 257)

    # Those will be filled in the loop:
    metrics_dict, distr_dict = [
        # Initialize dictionaries with empty lists
        {key: [] for key in d} for d in [metric_funcs, bins_dict]
    ]
    distr_dict['Pixel_Intensity'] = []

    # Calculate metrics for every time step.
    # Shapes:
    # metrics_dict --> {metric: (T, bsize)}
    # distr_dict --> {metric: (T, n_bins)}
    for batch in tqdm(
        batch_st.unbind(dim=1), 
        desc=f'Looping through t', 
        total=batch_st.shape[1]):
        for key, fnc in metric_funcs.items():
            # Clamp batch to [0, 1]
            batch = torch.clamp(batch, 0, 1)
            
            # Calculate metric
            metric_batch = batch_metric(batch, fnc)
            metrics_dict[key].append(metric_batch)

            # Calculate distribution
            distr_batch = np.histogram(metric_batch, bins=bins_dict[key])[0]
            distr_dict[key].append(distr_batch)
        
        # Pixel distribution
        pixel_dist = np.histogram(
            batch.squeeze().numpy(), bins=bins_dict['Pixel_Intensity']
        )[0]
        distr_dict['Pixel_Intensity'].append(pixel_dist)

    # Convert lists in dicts to numpy arrays
    metrics_dict, distr_dict = [
        {key: np.stack(val) for key, val in d.items()}
        for d in [metrics_dict, distr_dict]
    ]
    out = [metrics_dict, distr_dict, bins_dict]

    if pickle_file is not None:
        with open(pickle_file, 'wb') as f:
            pickle.dump(out, f)

    return out

    
def set_distribution_plot(counts, edges, ax,
                          label='', color="black", alpha=0.5, fill=False):
    # Extract counts and edges from distribution
    c_norm = norm(counts)
    c_norm_err = norm_err(counts)

    # Add number of images to label
    if label is not None:
        label += f" (n={int(np.sum(counts)):_})"

    # Plot distribution and error bars
    line = ax.stairs(c_norm, edges, alpha=alpha, fill=fill, label=label, color=color)
    scatter = ax.errorbar(centers(edges), c_norm, yerr=c_norm_err,
                alpha=0.75 * alpha, color=color, ls="none", elinewidth=0.5,
                capsize=2.5, capthick=0.5)

    # Set x limits
    xmax = max(edges)
    ax.set_xlim(-0.02 * xmax, 1.02*xmax)
    return line, scatter


def update_distribution_plot(counts, edges, line, scatter, ax, label=None):
    # Extract counts and edges from distribution
    c_norm = norm(counts)
    c_norm_err = norm_err(counts)

    # Update distribution and error bars
    line.set_data(c_norm, edges)
    if label is not None:
        label += f" (n={int(np.sum(counts)):_})"
        line.set_label(label)
    _, (bottoms, tops), (verts,) = scatter.lines
    x = centers(edges)
    y1 = c_norm - c_norm_err
    y2 = c_norm + c_norm_err
    bottoms.set_data(x, y1)
    tops.set_data(x, y2)
    segments = np.stack([
        np.stack([x, y1], axis=1), np.stack([x, y2], axis=1)
    ], axis=1)
    verts.set_segments(segments)

    # Set y limits
    set_y_limits(
        np.concatenate([c_norm + c_norm_err, c_norm - c_norm_err]), ax
    )

    return line, bottoms, tops, verts


def set_y_limits(counts, ax, data_present=True):
    y_min, y_max = ax.get_ylim()
    new_ymax = max(counts) * 1.02
    new_ymin = min(counts) * 0.98
    if data_present:
        new_ymax = max(new_ymax, y_max)
        new_ymin = min(new_ymin, y_min)
    ax.set_ylim(bottom=new_ymin, top=new_ymax)


def set_x_limits(counts, edges, ax, data_present=True):
    _, x_max = ax.get_xlim()
    new_xmax = max(edges[1:][counts > 0]) * 1.02
    if data_present:
        new_xmax = max(new_xmax, x_max)
    ax.set_xlim(-0.02 * new_xmax, new_xmax)

def create_animation(distr_dict, bins_dict):
    # For comparison
    target_distr = get_distributions_lofar()

    # Fix order for plot
    distr_dict = OrderedDict([
        (key, distr_dict[key])
        for key in ['Pixel_Intensity', 'Active_Pixels', 'Image_Mean']
    ])

    # Create figure and axes
    fig, axes = plt.subplots(3, 1, figsize=(10, 15), tight_layout=True)


    # Plot first (T=0) distributions to axes, return _ objects in list
    plot_items = []
    for ax, metric in zip(axes.flatten(), distr_dict.keys()):
        line_sched, scatter = set_distribution_plot(
            distr_dict[metric][0], bins_dict[metric], ax=ax,
            color='royalblue', label='Diffusion'
        )
        plot_items.append((line_sched, scatter))
        ax.set_xlabel(metric.replace('_', ' '))
        ax.set_yscale('log')
        # Target distribution
        add_distribution_plot(
            *target_distr[metric], ax, color='tab:orange', label='LOFAR'
        )
        ax.legend(ncols=2, loc='upper right')
    fig.suptitle(f'T = 0')

    def animate(i):
        out = []
        for item, metric, ax in zip(plot_items, distr_dict.keys(), axes.flatten()):
            line, scatter = item
            items_out = update_distribution_plot(
                distr_dict[metric][i], bins_dict[metric], 
                line, scatter, ax, label='Diffusion'
            )
            
            out += [*items_out]
            ax.legend(ncols=2, loc='upper right')
        fig.suptitle(f'T={i}')
        return out

    ani = animation.FuncAnimation(fig, animate, 
                                frames=26,
                                interval=200,
                                repeat_delay=2000,
                                blit=True)

    plt.rcParams['animation.embed_limit'] = 2**128
    plt.rcParams["animation.html"] = "jshtml"
    return ani

Calculate distributions & make animation:

In [ ]:
batch_st = torch.load(
    '/home/bbd0953/diffusion/analysis_results/EDM_SNR5_50as/EDM_SNR5_50as_samples_10000Imgs_T=25.pt',
    map_location='cpu'
)

In [ ]:
metrics_dict, distr_dict, bins_dict = get_metrics_and_distributions(batch_st)

In [ ]:
ani = create_animation(distr_dict, bins_dict)
ani

Save animation:

In [ ]:

ani.save(
    out_folder / f"{model_dir.name}_distribution_animation_{outfile_specifier}.mp4"
)

Distributions without animation:

In [ ]:
from utils.plot_utils import add_distribution_plot
from matplotlib import colormaps

def plot_distributions(batch_st_files, pickle_files=None, labels=None):
    # For comparison
    target_distr = get_distributions_lofar()
    metric_keys = ['Pixel_Intensity', 'Active_Pixels', 'Image_Mean']

    # Create figure and axes
    fig, axes = plt.subplots(3, 1, figsize=(10, 15), tight_layout=True)

    # Plot LOFAR & axes setup
    for ax, metric in zip(axes.flatten(), metric_keys):
            ax.set_xlabel(metric.replace('_', ' '))
            ax.set_yscale('log')
            # Target distribution
            add_distribution_plot(
                *target_distr[metric], ax, color='tab:orange', label='LOFAR'
            )

    if pickle_files is None:
        pickle_files = [None] * len(batch_st_files)
    if labels is None:
        labels = [None] * len(batch_st_files)

    cmap = 'tab10'
    colors = colormaps[cmap](np.linspace(0, 1, len(batch_st_files)))
    
    for st_file, pickle_file, label, color in zip(
        batch_st_files, pickle_files, labels, colors):
        batch_st = torch.load(st_file, map_location='cpu')
        _, distr_dict, bins_dict = get_metrics_and_distributions(
            batch_st, pickle_file
        )

        # Fix order for plot
        distr_dict = OrderedDict([
            (key, distr_dict[key])
            for key in metric_keys
        ])
        
        # Plot last (T=-1) distributions to axes
        for ax, metric in zip(axes.flatten(), distr_dict.keys()):
            add_distribution_plot(
                distr_dict[metric][-1], bins_dict[metric], ax=ax,
                color=color, label=label
            )
            ax.legend(ncols=2, loc='upper right')

    return fig, axes

In [ ]:
batch_st_files = [
    Path('/home/bbd0953/diffusion/analysis_results/EDM_Pmean=-2.5_Pstd=1.8/EDM_Pmean=-2.5_Pstd=1.8_samples_1000Imgs_T=25_sigma_min=1.00e-03_sigma_max=10_rho=10.pt')
]
batch_st_files.append(batch_st_files.pop(0))
pkl_files = [
    f.with_name(f.name.replace('samples', 'metrics')).with_suffix('.pkl')
    for f in batch_st_files
]

%matplotlib inline
plot_distributions(batch_st_files, pkl_files)

In [ ]:
batch_st_parent = Path('/home/bbd0953/diffusion/analysis_results/EDM_Pmean=-2.5_Pstd=1.8')
batch_st_files = sorted(batch_st_parent.glob('*rho=10.pt'))
batch_st_files.append(batch_st_files.pop(0))
pkl_files = [
    f.with_name(f.name.replace('samples', 'metrics')).with_suffix('.pkl')
    for f in batch_st_files
]
labels = [
    f'{n*20}k' for n in range(1, 5)
]

%matplotlib inline
plot_distributions(batch_st_files, pkl_files, labels)

In [ ]:
parent = Path(
    '/home/bbd0953/diffusion/analysis_results/fineTuning'
)

batch_st_files = sorted(parent.glob('*/*.pt'))
pkl_files = [
    f.with_name(f.name.replace('samples', 'metrics')).with_suffix('.pkl')
    for f in batch_st_files
]
labels = [
    f.parent.name.split('_')[-1] for f in batch_st_files
]

plot_distributions(batch_st_files, pickle_files=pkl_files, labels=labels)

## Images

Optionally load a different stack:

In [9]:
import numpy as np

payload = np.load(
    '/hs/fs08/data/group-brueggen/tmartinez/diffusion/analysis_results/icm_sampling_dev/inpainting_context_tdrop_variation_samples.npy',
    allow_pickle=True
).item()
batch_st = payload['samples']

Create animation:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import torch

# Number of images to be plotted on animation
side_length = 4
n_imgs = side_length**2

# Pick n_imgs random images from batch_st
# idx = torch.randint(0, batch_st.shape[0], (n_imgs,))

# Alternatively, set idx to a list of indices
# idx = [
#     985, 9303, 731, 7516,
#     9194, 7516, 2250, 1067,
#     3174, 6124, 4376,
#     3218, 3176, 3529, 7896
# ]

imgs = batch_st[idx].squeeze().numpy()
# imgs = np.clip(imgs, 0, 1)

# Create plot with n_imgs axes of quadratic shape, space for colorbars, no ticks
fig, axes = plt.subplots(
    # side_length, side_length, figsize=(15, 15)
    3, 5, figsize=(15, 10)
)
fig.subplots_adjust(hspace=0.1, wspace=0.1, left=0.05, right=0.95)
for ax in axes.flatten():
    ax.axis("off")

# Helper function to get color limit of image
get_clim = lambda img: (img.min(), img.max())
    
# Plot first image to axes, return imshow objects in list
ims = []
for ax, img in zip(axes.flatten(), imgs):
    min, max = get_clim(img[0])
    im = ax.imshow(img[0], vmin=min, vmax=max)
    ims.append(im)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f"T={0}")

def animate(i):
    for im, img in zip(ims, imgs):
        clim = get_clim(img[i])
        im.set_data(img[i])
        im.set_clim(*clim)
        im.axes.set_title(f"T={i}")
    return ims

%matplotlib inline
ani = animation.FuncAnimation(fig, animate, 
                              frames=26,
                              interval=300,
                              repeat_delay=2000,
                              blit=True);

plt.rcParams['animation.embed_limit'] = 2**128
# plt.rcParams.update(plt.rcParamsDefault)
plt.rcParams["animation.html"] = "jshtml"
# HTML(ani.to_jshtml())
ani

Save animation:

In [ ]:
out_folder = Path(
    '/home/bbd0953/diffusion/analysis_results/animations'
)

ani.save(
    out_folder / f"Diffusion_process_demo.mp4"
)

# Functions with Sliders

## Training Noise Distribution

In [ ]:
from scipy.stats import norm, lognorm
import numpy as np

def noise_schedule(T, sigma_min=2e-3, sigma_max=80, rho=7):
        # Time steps
        step_inds = np.arange(T)
        rho_inv = 1 / rho
        sigma_steps = (
            (sigma_max**rho_inv + step_inds / (T - 1)
             * (sigma_min**rho_inv - sigma_max**rho_inv))**rho
        )
        # sigma_steps = torch.cat([sigma_steps, torch.zeros_like(sigma_steps[:1])])  # t_N=0
        return step_inds, sigma_steps
    
def lognorm_pdf(x, P_mean, P_std):
    return norm.pdf(np.log(x), scale=P_std, loc=P_mean)

In [ ]:
from matplotlib.widgets import Button, Slider
import numpy as np

x = np.logspace(-4, 2, 10_000)
_, sigmas = noise_schedule(25, sigma_min=2e-3)
                       
# Initial parameters
P_mean = -1.2
P_std = 1.2

# Create the figure
fig, ax = plt.subplots()
ax.set_xscale('log')
ax.set_xlabel('Noise Level')

# Line to be manipulated
line_pdf, = ax.plot(x, lognorm_pdf(x, P_mean, P_std))

# Static lines:
ax.plot(x, lognorm_pdf(x, P_mean, P_std), color='green', alpha=0.5, ls='--')
for sigma in sigmas:
    ax.axvline(sigma, linestyle="--", color="red", alpha=0.5, linewidth=0.7)

# adjust the main plot to make room for the sliders
fig.subplots_adjust(bottom=0.25)

# Make a horizontal slider to control the parameters.
def add_slider(y_pos, **slider_kwargs):
    ax_param = fig.add_axes([0.25, y_pos, 0.65, 0.03])
    slider = Slider(
        ax=ax_param,
        **slider_kwargs
    )
    return slider

mean_slider = add_slider(
    0.15, label='P_mean', valmin=-5, valmax=5, valinit=P_mean, valstep=0.1
)
std_slider = add_slider(
    0.1, label='P_std', valmin=1e-4, valmax=5, valinit=P_std, valstep=0.1
)
sliders = [mean_slider, std_slider]

# The function to be called anytime a slider's value changes
def update(val):
    y = lognorm_pdf(x, mean_slider.val, std_slider.val)
    line_pdf.set_ydata(y)
    # Reset Y limit
    ax.set_ylim(top=np.max(y)*1.05)
    fig.canvas.draw_idle()
    


# register the update function with each slider
[slider.on_changed(update) for slider in sliders]

# Create a `matplotlib.widgets.Button` to reset the sliders to initial values.
resetax = fig.add_axes([0.8, 0.025, 0.1, 0.04])
button = Button(resetax, 'Reset', hovercolor='0.975')

def reset(event):
    [slider.reset() for slider in sliders]
button.on_clicked(reset)

%matplotlib widget
fig.show()

## Sampling Noise Levels

In [ ]:
import numpy as np

def noise_schedule(T, sigma_min=2e-3, sigma_max=80, rho=7):
        # Time steps
        step_inds = np.arange(T)
        rho_inv = 1 / rho
        sigma_steps = (
            (sigma_max**rho_inv + step_inds / (T - 1)
             * (sigma_min**rho_inv - sigma_max**rho_inv))**rho
        )
        # sigma_steps = torch.cat([sigma_steps, torch.zeros_like(sigma_steps[:1])])  # t_N=0
        return step_inds, sigma_steps

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.widgets import Button, Slider, CheckButtons


t = np.linspace(0, 1, 1000)

# Define initial parameters
init_T = 25
init_sigma_min = 2e-3
init_sigma_max = 80
init_rho = 7

# Create the figure and the line that we will manipulate
fig, ax = plt.subplots()
x, y = noise_schedule(
                    init_T,
                    init_sigma_min,
                    init_sigma_max,
                    init_rho
                )
line_sched = ax.scatter(x, y, lw=2)
line_dt = ax.scatter(x[:-1], np.abs(np.diff(y)), lw=2)

ax.set_xlabel('Time Step')
ax.grid(alpha=0.3)

# adjust the main plot to make room for the sliders
fig.subplots_adjust(bottom=0.25)

# Make a horizontal slider to control the parameters.
def add_slider(y_pos, **slider_kwargs):
    ax_param = fig.add_axes([0.25, y_pos, 0.35, 0.03])
    slider = Slider(
        ax=ax_param,
        **slider_kwargs
    )
    return slider

T_slider = add_slider(
    0.15, label='T', valmin=2, valmax=50, valinit=init_T, valstep=1,
)
sigma_min_slider = add_slider(
    0.1, label='sigma_min', valmin=1e-3, valmax=25, valinit=init_sigma_min
)
sigma_max_slider = add_slider(
    0.05, label='sigma_max', valmin=0.1, valmax=100, valinit=init_sigma_max
)
rho_slider = add_slider(
    0, label='rho', valmin=1, valmax=10, valinit=init_rho
)


# The function to be called anytime a slider's value changes
def update(val):
    x, y = noise_schedule(
        T_slider.val,
        sigma_min_slider.val,
        sigma_max_slider.val,
        rho_slider.val
    )
    line_sched.set_offsets(np.c_[x, y])
    line_dt.set_offsets(np.c_[x[:-1], np.abs(np.diff(y))])
    # Reset axis limits
    ax.set_xlim(- x.max()*0.05, x.max()*1.05)
    ax.set_ylim(-y.max()*0.05, y.max()*1.05)
    fig.canvas.draw_idle()


# register the update function with each slider
T_slider.on_changed(update)
sigma_min_slider.on_changed(update)
sigma_max_slider.on_changed(update)
rho_slider.on_changed(update)


# Create a `matplotlib.widgets.Button` to reset the sliders to initial values.
resetax = fig.add_axes([0.8, 0.025, 0.1, 0.04])
button = Button(resetax, 'Reset', hovercolor='0.975')

def reset(event):
    T_slider.reset()
    sigma_min_slider.reset()
    sigma_max_slider.reset()
    rho_slider.reset()
button.on_clicked(reset)

# Create button for y-axis log scale
rax = fig.add_axes([0.8, 0.065, 0.1, 0.04])
log_button = Button(rax, 'Y log', hovercolor='0.975')
def ylog(event):
    current = ax.get_yscale()
    ax.set_yscale('log' if current == 'linear' else 'linear')
log_button.on_clicked(ylog)


%matplotlib widget
fig.show()

In [ ]:
fig, ax = plt.subplots()
scatter = ax.scatter(*noise_schedule(
                    init_T,
                    init_sigma_min,
                    init_sigma_max,
                    init_rho
                ), lw=2)

In [ ]:
scatter